# RBAC, Security, Troubleshooting & CKA Prep

## What's covered

- **Identity in Kubernetes** — humans (Users) vs workloads (ServiceAccounts), how each authenticates
- **ServiceAccounts** — pod identity, automounted tokens, the projected ServiceAccountToken
- **RBAC** — `Role`, `ClusterRole`, `RoleBinding`, `ClusterRoleBinding`; verbs, resources, subjects
- **Aggregated roles** and the well-known defaults (`view`, `edit`, `admin`, `cluster-admin`)
- **`kubectl auth can-i`** — the fastest "did my RBAC rule actually do what I wanted?" check
- **Pod security context** — `runAsUser`, `runAsNonRoot`, `fsGroup`, `capabilities`, read-only root, seccomp
- **Pod Security Admission** — the replacement for the deprecated PodSecurityPolicy, with its three levels (`privileged`, `baseline`, `restricted`)
- **etcd encryption at rest** — `EncryptionConfiguration`, the most-CKA'd "extra credit" cluster-hardening step
- **Image security and supply chain** — private registries, image signing (sigstore, cosign), the `imagePullPolicy` gotcha
- **Troubleshooting playbooks** — application failures, control plane failures, worker node failures, networking failures, in the order the CKA tests them
- **`kubectl` power tools** — `explain`, `--dry-run=client`, `debug`, `top`, `auth can-i`
- **Exam strategy** — what to drill, how to spend the two-hour budget, the tricks worth remembering

By the end of this notebook you should be able to write any RBAC role from scratch in under a minute, harden a pod's security context with production-grade defaults, identify *why* a pod or node or Service is broken without flailing, and walk into the CKA with a punch-list of muscle-memory you can lean on under time pressure.

Notebooks 02 through 09 are the prerequisites — this is the chapter that pulls everything together and adds the security-and-troubleshooting layer the exam grades heavily.

## Identity in Kubernetes — Users vs ServiceAccounts

Two kinds of identity exist in a cluster, and they're easy to mix up.

### Users — for humans (and CI systems)

A **User** is whatever the API server figures out from your kubeconfig:

- A client certificate's `Subject: CN=alice, O=engineering` — `alice` is the user, `engineering` is a group.
- An OIDC ID token's `sub:` claim.
- A static token in a tokenfile, mapped to a username.
- A webhook auth response.

**Users are not Kubernetes objects.** There's no `kubectl create user alice`. The API server doesn't store them, doesn't authenticate them itself — it just *recognises* the identities your authenticator chain returns. Add a user by issuing a new client cert (signed by the cluster CA) or by configuring an OIDC provider.

### ServiceAccounts — for pods

A **ServiceAccount** is a Kubernetes object that exists in the API server, namespaced, with a name like `default/api-server` (notebook 08's brief mention). Pods reference a ServiceAccount by name; the kubelet mounts a token for that SA into the pod; the pod uses that token to call the API server.

```yaml
apiVersion: v1
kind: ServiceAccount
metadata:
  name: api-server
  namespace: default
```

ServiceAccounts are how *workloads* authenticate. Users are how *people* authenticate. The API server can apply RBAC to either — same rules, same subjects-list syntax.

### The default SA

Every namespace gets a `default` ServiceAccount, and every pod that doesn't specify one uses it. The default SA has *no permissions* — RBAC bindings to it are zero by default. Don't grant `default` more than it needs; create a workload-specific SA.

### Groups

Both users and ServiceAccounts have a notion of *groups*:

- **Users** belong to groups parsed from the cert's `O=` fields or the OIDC `groups` claim.
- **ServiceAccounts** belong to four automatic groups:
  - `system:serviceaccount:<namespace>:<name>` — this exact SA.
  - `system:serviceaccounts:<namespace>` — every SA in the namespace.
  - `system:serviceaccounts` — every SA in the cluster.
  - `system:authenticated` — any authenticated identity.

That last one matters: a `ClusterRoleBinding` to `system:authenticated` grants everybody — including every SA. Use sparingly.

## ServiceAccounts — pod identity

### What the kubelet mounts

When a pod is scheduled, the kubelet asks the API server's `TokenRequest` API for a **projected JWT token** for the pod's ServiceAccount. It mounts three files into the pod at `/var/run/secrets/kubernetes.io/serviceaccount/`:

```
ca.crt       # the API server's CA — for verifying TLS
namespace    # the pod's namespace, as a string
token        # the JWT, signed by the cluster's token signing key
```

In-cluster API clients (the official client libraries, `kubectl` from inside a pod) read these three files to construct an authenticated connection.

### Modern (bound) tokens

The tokens the kubelet projects today have:

- **Expiration** — typically 1 hour; the kubelet rotates them automatically.
- **Audiences** — the token is only valid for `kubernetes.default.svc` and similar.
- **A binding to the pod** — if the pod's deleted, the token's revoked.

Legacy "long-lived SA tokens" (a Secret per SA, never expiring) are still creatable in older clusters but have been phased out. Treat any SA token Secret older than Kubernetes 1.24 as suspicious.

### `automountServiceAccountToken: false`

By default the kubelet mounts a token into every pod. For workloads that don't talk to the API server, mount nothing:

```yaml
spec:
  serviceAccountName: default
  automountServiceAccountToken: false   # skip the mount entirely
```

This is best practice for pure web servers, database engines, anything that doesn't call the API server. Reduces blast radius.

### Image pull secrets via SA (notebook 05 recap)

A registry credential attached to a ServiceAccount applies automatically to every pod that uses it:

```bash
kubectl patch serviceaccount default -p '{"imagePullSecrets":[{"name":"regcred"}]}'
```

No per-pod `imagePullSecrets` needed.

In [ ]:
# Create a ServiceAccount and a pod that uses it
!kubectl create serviceaccount demo-sa
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: sa-demo
spec:
  serviceAccountName: demo-sa
  containers:
    - name: app
      image: busybox:1.36
      command: ["sleep", "3600"]
EOF
!sleep 6
# What the kubelet mounted for the SA
!kubectl exec sa-demo -- ls /var/run/secrets/kubernetes.io/serviceaccount/
!echo '---'
# The first ~80 chars of the JWT — proves it's a real token without dumping the whole thing
!kubectl exec sa-demo -- sh -c 'head -c 80 /var/run/secrets/kubernetes.io/serviceaccount/token; echo'
!echo '---'
# Who is the pod, from the API server's perspective?
# The default SA has NO permissions; this call will be denied 403 — which is the lesson.
!kubectl exec sa-demo -- sh -c '
  TOKEN=$(cat /var/run/secrets/kubernetes.io/serviceaccount/token)
  wget -q -O- --no-check-certificate \
    --header="Authorization: Bearer $TOKEN" \
    https://kubernetes.default.svc/api/v1/namespaces/default/pods/sa-demo 2>&1 | head -5' || \
  echo "(403 from the API server — demo-sa has no RBAC binding yet)" 

## RBAC — Roles and ClusterRoles

**RBAC** (Role-Based Access Control) is Kubernetes' permission system. It answers the AUTHZ question from notebook 08: *can this subject do this verb on this resource?*

Two kinds of "role" objects, identical in shape, different in scope:

- **`Role`** — namespaced. Grants permissions on resources *in one namespace*.
- **`ClusterRole`** — cluster-scoped. Grants permissions on:
  - cluster-scoped resources (nodes, PVs, namespaces themselves),
  - non-resource URLs (`/healthz`, `/metrics`),
  - resources in *every* namespace (when used with a `ClusterRoleBinding`).

```yaml
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  name: pod-reader
  namespace: team-a
rules:
  - apiGroups: [""]
    resources: ["pods"]
    verbs: ["get", "list", "watch"]
  - apiGroups: [""]
    resources: ["pods/log", "pods/status"]
    verbs: ["get"]
  - apiGroups: ["apps"]
    resources: ["deployments"]
    verbs: ["get", "list"]
```

Three parts to each rule:

- **`apiGroups`** — `""` (the empty string) means the core group (`Pod`, `Service`, `ConfigMap`, `Namespace`, ...). Other examples: `apps` (Deployments, StatefulSets), `batch` (Jobs, CronJobs), `networking.k8s.io` (Ingress, NetworkPolicy), `rbac.authorization.k8s.io` (Roles, Bindings).
- **`resources`** — the lowercase plural kind. `pods`, `deployments`, `secrets`. Sub-resources use `/`: `pods/log`, `pods/exec`, `deployments/scale`.
- **`verbs`** — `get`, `list`, `watch`, `create`, `update`, `patch`, `delete`, `deletecollection`. Plus a few special ones: `*` (everything), `impersonate` (act as another user), `bind` / `escalate` (RBAC-specific).

### Common verb combinations

| Use case | Verbs |
|---|---|
| Read-only access | `get`, `list`, `watch` |
| Read and modify | `get`, `list`, `watch`, `create`, `update`, `patch` |
| Full control | `get`, `list`, `watch`, `create`, `update`, `patch`, `delete` |
| Delete-by-label-selector | add `deletecollection` |
| Run `kubectl exec` / `logs` | `get` on `pods/exec` and `pods/log` |

### The "no deny" rule

RBAC is **purely additive**. There's no way to write "deny X". Permissions accumulate from every binding that names you. To remove a permission, you delete the binding that grants it.

## RBAC — bindings, subjects, and `auth can-i`

A `Role` (or `ClusterRole`) is *just permissions*. It doesn't apply to anyone until you **bind** it to a subject.

### `RoleBinding` and `ClusterRoleBinding`

- **`RoleBinding`** — namespace-scoped. Binds a Role *or* a ClusterRole to subjects in one namespace.
- **`ClusterRoleBinding`** — cluster-scoped. Binds a ClusterRole to subjects across the whole cluster.

```yaml
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: alice-reads-team-a
  namespace: team-a
subjects:
  - kind: User
    name: alice                         # User name from the authenticator
    apiGroup: rbac.authorization.k8s.io
  - kind: ServiceAccount
    name: ci-runner
    namespace: ci
  - kind: Group
    name: system:serviceaccounts:team-a # all SAs in team-a
    apiGroup: rbac.authorization.k8s.io
roleRef:
  kind: Role
  name: pod-reader
  apiGroup: rbac.authorization.k8s.io
```

Each subject is one of three kinds:

- **`User`** — `kind: User`, with the username your authenticator returned.
- **`ServiceAccount`** — `kind: ServiceAccount`, with `name` and `namespace`.
- **`Group`** — `kind: Group`, with the group name.

### The Role / ClusterRole crossover

A `RoleBinding` can bind a `ClusterRole` (not just a `Role`). When it does, the ClusterRole's permissions only apply *in that binding's namespace*. This is the standard trick for "reuse one ClusterRole across many namespaces":

```yaml
# Define once, cluster-scoped
kind: ClusterRole
metadata:
  name: pod-reader
rules: [...]
---
# Bind in each namespace
kind: RoleBinding
metadata:
  name: team-a-pod-reader
  namespace: team-a
roleRef:
  kind: ClusterRole         # the ClusterRole defines the permissions
  name: pod-reader
subjects: [...]
```

### `kubectl auth can-i` — the verification tool

The single most useful RBAC command. Asks the API server "would this verb on this resource be allowed for me, right now?":

```bash
kubectl auth can-i list pods                          # for yourself in current namespace
kubectl auth can-i list pods -n team-a                # specify namespace
kubectl auth can-i create deployments --as alice      # impersonate (needs `impersonate` perm)
kubectl auth can-i '*' '*' --all-namespaces           # am I cluster-admin?
```

Returns `yes` / `no` / `yes (and a warning)` and an exit code. **The CKA expects you to use this all the time.**

### Imperative creation

Quickest way to get a Role and Binding for the exam:

```bash
kubectl create role pod-reader \
  --verb=get,list,watch --resource=pods,pods/log

kubectl create rolebinding alice-can-read \
  --role=pod-reader --user=alice
```

`kubectl create clusterrole` and `kubectl create clusterrolebinding` have the same shape. Add `--dry-run=client -o yaml` if you want a YAML template to edit.

In [ ]:
# A SA, a Role, and a RoleBinding — all imperatively for speed
!kubectl create serviceaccount reader-sa
!kubectl create role pod-reader --verb=get,list,watch --resource=pods
!kubectl create rolebinding reader-sa-pod-reader --role=pod-reader --serviceaccount=default:reader-sa
!echo '---'
# Can this SA list pods? --as= lets you impersonate
!kubectl auth can-i list pods --as=system:serviceaccount:default:reader-sa
!echo '---'
# Can it list secrets? (it shouldn't)
!kubectl auth can-i list secrets --as=system:serviceaccount:default:reader-sa
!echo '---'
# Can it delete pods? (no — only get/list/watch were granted)
!kubectl auth can-i delete pods --as=system:serviceaccount:default:reader-sa
!echo '---'
# Now bind the built-in 'view' ClusterRole to verify cross-resource read
!kubectl create rolebinding reader-sa-view --clusterrole=view --serviceaccount=default:reader-sa
!kubectl auth can-i get configmaps --as=system:serviceaccount:default:reader-sa
!kubectl auth can-i get secrets --as=system:serviceaccount:default:reader-sa

## Aggregated and default roles

Kubernetes ships several `ClusterRole`s you can rely on by name. The big four:

| ClusterRole | Permissions |
|---|---|
| **`cluster-admin`** | Everything, everywhere. The root role. |
| **`admin`** | Full control of a namespace's resources, *but cannot modify the namespace itself or its quotas*. Granted via `RoleBinding`. |
| **`edit`** | Read-write on workloads (Pods, Deployments, Services, ConfigMaps, Secrets); *cannot* manage RBAC. |
| **`view`** | Read-only on most workloads (excluding Secrets). |

You'll see these in every cluster — `kubectl get clusterroles | grep -E 'admin|edit|view|cluster-admin'`. The standard pattern is `kubectl create rolebinding team-a-admins --clusterrole=admin --group=team-a-admins` — gives a whole group namespace-admin in one namespace.

### Aggregation — composing ClusterRoles

A ClusterRole can be defined as the *union* of other ClusterRoles, via labels. The well-known `view`, `edit`, `admin` roles are all aggregated — you can extend them by adding a new ClusterRole with the right label:

```yaml
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRole
metadata:
  name: monitoring-view
  labels:
    rbac.authorization.k8s.io/aggregate-to-view: "true"     # gets folded into "view"
rules:
  - apiGroups: ["monitoring.coreos.com"]
    resources: ["servicemonitors", "podmonitors"]
    verbs: ["get", "list", "watch"]
```

After this is applied, anyone bound to `view` automatically gets read access to ServiceMonitors and PodMonitors too. The control-plane's RBAC controller watches for the labels and re-aggregates.

### The `system:` reserved roles

Many `system:*` ClusterRoles ship with the cluster:

- `system:basic-user`, `system:discovery` — anonymous and authenticated discovery.
- `system:node` — the role the kubelet uses for its node-side calls.
- `system:kube-controller-manager`, `system:kube-scheduler` — per-component.

Don't bind directly to these. They're built-ins the control plane manages.

### Granting `view` excludes Secrets — for a reason

The `view` ClusterRole *deliberately* excludes Secrets. If you bind someone to `view` and they ask "why can't I read Secrets?", the answer is "you shouldn't be able to — Secret data is high-trust." If they truly need Secret access, give them a separate, narrower role.

## Pod security context — running with the right user, perms, and capabilities

Containers by default run as `root` inside their pod, with all the Linux capabilities the runtime allows. Both are dangerous. The **security context** is your toolkit for hardening.

```yaml
spec:
  securityContext:                # pod-level
    runAsNonRoot: true
    runAsUser: 1000
    runAsGroup: 3000
    fsGroup: 2000
    seccompProfile:
      type: RuntimeDefault
  containers:
    - name: app
      image: ...
      securityContext:            # container-level overrides pod-level
        allowPrivilegeEscalation: false
        readOnlyRootFilesystem: true
        runAsUser: 1001
        capabilities:
          drop: ["ALL"]
          add: ["NET_BIND_SERVICE"]
```

Field by field:

- **`runAsUser` / `runAsGroup`** — the UID and GID the container's first process runs as. Numbers, not names — the kernel only knows numbers.
- **`runAsNonRoot: true`** — the container fails to start if it tries to run as UID 0. The cheapest safety net. Use it.
- **`fsGroup`** — the GID applied to all files in the pod's mounted volumes. Lets a non-root container actually write to a volume.
- **`allowPrivilegeEscalation: false`** — equivalent to `setuid` / `setgid` being disabled. Combined with `runAsNonRoot`, the container can't get root by tricks.
- **`readOnlyRootFilesystem: true`** — `/` is mounted read-only. Anything that needs to write needs an `emptyDir` mount at the right path. Killer of "exploit drops a binary in `/tmp`" attacks.
- **`capabilities.drop: ["ALL"]`** — drop every Linux capability the runtime would have granted. Combine with `add` for the few you actually need.
- **`seccompProfile.type: RuntimeDefault`** — apply the runtime's default seccomp filter, which blocks dangerous syscalls. Set `Unconfined` only if you really need it.

### Privileged pods

`privileged: true` on a container's security context turns off all the restrictions: the container has full root on the node, can write to host devices, can load kernel modules. **Only DaemonSets that need to manage the node** (CNI agents, CSI drivers, some monitoring) should ever set this. Restrict via Pod Security Admission (next).

### `runAsNonRoot` + non-root images

A common trip-up: you set `runAsNonRoot: true` but the image's default user is root. Either set `runAsUser: <something-not-0>` in the pod spec, or build the image with a `USER 1000` line in the Dockerfile. Many official images now ship as non-root by default — modern Bitnami, Chainguard, distroless images.

In [ ]:
# A pod with a hardened security context
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Pod
metadata:
  name: hardened
spec:
  securityContext:
    runAsNonRoot: true
    runAsUser: 1000
    fsGroup: 1000
    seccompProfile:
      type: RuntimeDefault
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "id; mount | grep ' / ' | head -1; sleep 3600"]
      securityContext:
        allowPrivilegeEscalation: false
        readOnlyRootFilesystem: true
        capabilities:
          drop: ["ALL"]
      volumeMounts:
        - name: scratch
          mountPath: /tmp
  volumes:
    - name: scratch
      emptyDir: {}
EOF
!sleep 5
# The container is running as UID 1000 with a read-only root
!kubectl logs hardened
!echo '---'
# Writing outside /tmp fails (readOnlyRootFilesystem)
!kubectl exec hardened -- sh -c 'touch /forbidden 2>&1 || echo "(blocked, as expected)"'
!kubectl exec hardened -- sh -c 'touch /tmp/allowed && echo "(/tmp writable, as expected)"'

## Pod Security Admission (PSA) — enforcing security at the namespace level

Setting a security context on each pod is great, but you want a *blanket policy* that rejects pods which don't meet the bar. That's what **Pod Security Admission** does. It replaced `PodSecurityPolicy` (deprecated and removed in 1.25).

PSA defines three **levels** of policy, applied per-namespace via labels:

| Level | What's allowed |
|---|---|
| **`privileged`** | Everything. No restrictions. Default for `kube-system` and friends. |
| **`baseline`** | Sane defaults — no `privileged: true`, no host namespaces, no `hostPath`, no `NET_ADMIN`. Good for general workloads. |
| **`restricted`** | Hardest line — must be non-root, must drop all capabilities, must use seccomp, no privilege escalation. Match for high-trust workloads. |

And three **modes** that decide what happens when a pod violates the level:

| Mode | Behaviour |
|---|---|
| **`enforce`** | Reject the pod at admission. |
| **`audit`** | Allow but write an audit event. |
| **`warn`** | Allow but return a warning to the user (visible in `kubectl apply` output). |

You apply these by labelling the namespace:

```yaml
apiVersion: v1
kind: Namespace
metadata:
  name: secure
  labels:
    pod-security.kubernetes.io/enforce: restricted
    pod-security.kubernetes.io/enforce-version: latest
    pod-security.kubernetes.io/warn: restricted
    pod-security.kubernetes.io/audit: restricted
```

After applying, any pod created in `secure` that doesn't meet the `restricted` bar is rejected with an explanation listing which checks failed.

### The migration recipe

Common pattern for adopting PSA on an existing cluster:

1. Start with `warn` and `audit` at the target level. Existing pods aren't blocked but you see what would break.
2. Fix the pods.
3. Switch to `enforce`.

This avoids breaking production when you turn the switch.

### Custom admission with Gatekeeper / Kyverno

PSA's three levels are coarse-grained. For finer rules ("no Deployments without a `team` label", "all Ingresses must terminate TLS", "deny images from unapproved registries"), the standard tools are:

- **OPA Gatekeeper** — Rego-based policy engine.
- **Kyverno** — Kubernetes-native policy in YAML.

Both implement validating (and mutating) admission webhooks. Out of scope for the CKA; standard for production-grade clusters.

## etcd encryption at rest

Out of the box, Secret data in `etcd` is base64-encoded but **not encrypted** (notebook 05). Anyone with read access to the `etcd` data directory — a node attacker, an old backup snapshot — can read every Secret in plaintext.

The fix is an **`EncryptionConfiguration`** read by the API server, which encrypts specific resource types before writing them to `etcd`.

```yaml
# /etc/kubernetes/encryption-config.yaml
apiVersion: apiserver.config.k8s.io/v1
kind: EncryptionConfiguration
resources:
  - resources:
      - secrets
      - configmaps
    providers:
      - aescbc:
          keys:
            - name: key1
              secret: <base64 32-byte key>
      - identity: {}             # fallback — read access to unencrypted data
```

Then pass `--encryption-provider-config=/etc/kubernetes/encryption-config.yaml` to the API server (in the static-pod manifest from notebook 08).

### Provider ordering matters

The first provider in the list is used for **encrypting writes**. All providers can be used for **decrypting reads**. So during a key rotation:

1. Add the new key as the first provider, old key as the second. Restart the API server.
2. Re-encrypt existing data: `kubectl get secrets -A -o json | kubectl replace -f -` — this triggers each Secret to be re-written, picking up the new encryption.
3. Remove the old key from the config. Restart the API server.

`identity: {}` is the "no encryption" provider — keep it last during migration so existing unencrypted Secrets stay readable.

### What gets encrypted

You list resource types in `resources[].resources`. The most common targets:

- `secrets` — essential.
- `configmaps` — if you (unwisely) put credentials there.
- Custom resources where the CRD allows.

Pods, Services, Nodes — usually not encrypted; the data isn't sensitive in the same way.

### KMS providers

For the strongest setup, instead of a static key in `aescbc`, use a **KMS provider** that talks to an external HSM or KMS:

- `kms` provider type, points at a Unix socket where a KMS plugin listens.
- Plugins exist for AWS KMS, GCP KMS, Azure Key Vault, HashiCorp Vault.

The actual encryption key never lives in `etcd` or on the API server's disk. Out of scope for the CKA; common in production.

### What this is NOT

`EncryptionConfiguration` only protects data **at rest in `etcd`**. It does *not* encrypt:

- Secrets in transit to the kubelet (already protected by TLS).
- Secrets mounted into pod filesystems (visible to anyone with pod access).
- Backups that include the unencrypted `etcd` snapshot (use `--data-dir`-level encryption for those).

## Image security and supply chain

The CKA touches image security lightly; production clusters care a lot.

### Trust the image's digest, not its tag

A tag (`nginx:1.27-alpine`) is a moving label — the registry can rewrite it at any time. A digest (`nginx@sha256:abc...`) is an immutable content hash. Use digests in production manifests; use tags in development:

```yaml
image: nginx@sha256:abcdef0123456789...
```

The CKA-relevant flag: `imagePullPolicy` defaults to `Always` for `:latest` and unset tags (the registry is contacted on every pod start), and to `IfNotPresent` for specific tags. Using digests gets you immutability without the latency cost.

### Private registries — recap

Notebook 05's `kubectl create secret docker-registry` plus pod's `imagePullSecrets` field, or attached to the ServiceAccount. The kubelet uses the credential when pulling.

For cloud-managed clusters (EKS, GKE, AKS), the node's IAM identity can be granted registry access directly — no `imagePullSecret` needed.

### Image signing — sigstore and cosign

Two community standards:

- **Sigstore** — a project umbrella for keyless code signing using OIDC identities. `cosign` is the CLI.
- **Notation** — the OCI-standardised image signing toolchain.

Signed images are verified at admission time by tools like **Connaisseur** or **Kyverno**. Pods running unsigned images can be rejected.

### Vulnerability scanning

Out of scope for the cluster's admission layer, but core to your supply chain:

- **Trivy** — open-source scanner, runs locally, integrates with CI.
- **Snyk**, **Anchore**, cloud-native scanners (ECR scan-on-push, GCR Container Analysis).

In the CKA you won't be asked to set this up, but you should know "scan images before they enter the cluster" is the responsibility of your CI pipeline, not the cluster.

### Runtime security

- **Falco** — eBPF-based runtime detection. Watches syscalls in pods, alerts on suspicious patterns (a web server shouldn't be `exec`ing `bash`).
- **AppArmor and SELinux profiles** — per-container kernel-level confinement, configured via security context.

Both are out of scope for the CKA but standard for production.

## Troubleshooting — application failures

About 30% of the CKA is troubleshooting. The exam gives you a broken cluster; you fix it. The fastest path is to *know the failure modes* and the first commands to run for each.

### Pod is `Pending`

```bash
kubectl describe pod <name>     # the Events section is where the answer lives
```

Common reasons:

- **`FailedScheduling`** — no node fits. Read the reason: insufficient CPU/memory, untolerated taint, node selector mismatch, volume zone affinity unsatisfied.
- **`FailedAttachVolume` / `FailedMount`** — PVC isn't binding, or the volume is on a different node.
- **`ImagePullBackOff` / `ErrImagePull`** — image doesn't exist or the pull secret is wrong.

### Pod is `Running` but the container keeps restarting (`CrashLoopBackOff`)

```bash
kubectl logs <name>                    # current container — usually empty if it crashed fast
kubectl logs <name> --previous         # the PREVIOUS container's logs — this is what crashed
kubectl describe pod <name>            # check restartCount + last exit code
kubectl exec <name> -- sh              # if it's running long enough, jump in
```

Common reasons:

- **Application bug** — usually visible in logs.
- **OOMKilled** — exit code 137; the container exceeded its memory limit. `kubectl describe` shows `Reason: OOMKilled` in the last state.
- **Missing config** — a ConfigMap or Secret reference is broken.
- **Failing readiness probe** — the container doesn't restart, but the Service won't route. Check `kubectl get pod` for `READY: 0/1`.

### Pod can't reach a Service

```bash
kubectl get svc <name>                 # does the Service exist?
kubectl get endpoints <name>           # do endpoints exist? <none> means selector mismatch
kubectl describe svc <name>            # selector + endpoint addresses side-by-side
```

Then from a sibling pod:

```bash
nslookup <svc-name>                    # does DNS resolve?
nc -vz <svc-name> <port>               # is the port open?
```

### `kubectl debug` — ephemeral debug containers

When the pod is too minimal to inspect (distroless, scratch), attach an ephemeral debug container:

```bash
kubectl debug -it <pod> --image=busybox:1.36 --target=<container>
```

The debug container shares the target pod's process namespace; you can `ps`, `cat /proc/PID/...`, run network tools. The debug container is gone when you exit.

For node-level debug — drop into the node itself:

```bash
kubectl debug node/<name> -it --image=busybox:1.36
```

Mounts the node's root at `/host` inside the debug pod. Useful when you need to read kubelet logs, `/etc/kubernetes/manifests/`, etc.

## Troubleshooting — control plane failures

### Symptom: `kubectl` returns "connection refused" or "i/o timeout"

The API server isn't reachable. Either it's down, or your kubeconfig points at the wrong place.

```bash
# From the control-plane node:
sudo crictl ps | grep apiserver         # is the static pod running?
sudo journalctl -u kubelet -n 50        # kubelet logs — they show why static pods failed
ls /etc/kubernetes/manifests/           # are the manifests still there?
```

Common causes:

- **Bad flag in a static-pod manifest** — you edited `/etc/kubernetes/manifests/kube-apiserver.yaml` and broke it. The kubelet logs are explicit.
- **Cert expired** — see notebook 08. `kubeadm certs check-expiration` from the node.
- **etcd down** — the API server can't start without etcd.

### Symptom: pods stuck `ContainerCreating` cluster-wide

Usually one of:

- **CNI broken** — `kubectl get pods -n kube-system | grep -E 'calico|cilium|flannel'` to see if CNI pods are healthy. Pod logs from those pods tell you why.
- **kubelet can't reach the API server** — partial control-plane outage. The kubelet keeps retrying.

### Symptom: nodes `NotReady`

```bash
kubectl describe node <name>            # Conditions section
ssh <node>                              # then:
sudo systemctl status kubelet           # is the kubelet running?
sudo journalctl -u kubelet -n 100       # what's it complaining about?
```

Common kubelet failures:

- **Can't reach API server** — wrong `--kubeconfig`, expired cert, firewall.
- **Container runtime down** — `sudo systemctl status containerd`.
- **Disk pressure** — `df -h /var/lib/{docker,kubelet,containerd}`. Full disk flips the node to `NotReady`.

### Symptom: scheduler / controller-manager not making decisions

Check their logs from the static pod:

```bash
kubectl logs -n kube-system kube-scheduler-<node>
kubectl logs -n kube-system kube-controller-manager-<node>
```

Common: leader-election lease lost (look for `failed to acquire lease` lines), or RBAC permission missing.

### Symptom: API server slow

Profile what's calling it:

```bash
kubectl get --raw '/metrics' | grep apiserver_request_duration_seconds
```

Often the culprit is a noisy client (a runaway controller polling instead of watching) or etcd being slow.

## Troubleshooting — worker node failures

### Step 1 — what does the cluster see?

```bash
kubectl get nodes -o wide
kubectl describe node <name>
```

Read the `Conditions` block. Five matter:

- **`Ready`** — overall health.
- **`MemoryPressure`**, **`DiskPressure`**, **`PIDPressure`** — eviction triggers.
- **`NetworkUnavailable`** — CNI hasn't configured the node.

Read the `Events` block at the bottom — taints applied, evictions, system OOMs.

### Step 2 — SSH to the node, check the kubelet

```bash
sudo systemctl status kubelet
sudo journalctl -u kubelet -n 200 --no-pager
```

The kubelet logs every probe failure, every container start, every eviction. Read them backwards from "now" until you find the first complaint.

### Step 3 — check the container runtime

```bash
sudo crictl ps                          # running containers, via CRI
sudo crictl images                      # cached images
sudo systemctl status containerd        # or crio
sudo journalctl -u containerd -n 100
```

`crictl` is the cri-tools utility — like `docker ps` but speaks CRI directly. Useful when the kubelet is down but you want to see what's running.

### Step 4 — check disk

```bash
df -h
sudo du -sh /var/lib/containerd /var/lib/kubelet /var/log/pods
```

Big offender: log rotation broken. The kubelet keeps the last N container logs per pod, but a misconfigured logging setup can fill the disk.

### Step 5 — check network

```bash
ip addr                                 # is the CNI interface present?
ip route                                # are the pod-CIDR routes there?
iptables -t nat -L KUBE-SERVICES -n     # are kube-proxy rules installed?
sudo crictl logs <cni-agent-pod-id>     # CNI agent's own logs
```

### Common quick fixes

- **Cordon and drain the node** for maintenance: `kubectl cordon <node>` then `kubectl drain <node> --ignore-daemonsets`.
- **Uncordon when fixed**: `kubectl uncordon <node>`.
- **Force a kubelet restart**: `sudo systemctl restart kubelet`.
- **Recreate a single kube-proxy pod**: `kubectl delete pod -n kube-system <kube-proxy-pod>` (the DaemonSet replaces it).

## Troubleshooting — networking failures

### "Service is unreachable from a pod"

The three-question flowchart:

1. **Does DNS resolve?** From inside a test pod: `nslookup <svc>`. If no, the problem is CoreDNS or the pod's `dnsPolicy` / `resolv.conf`.
2. **Is the Service backed by endpoints?** `kubectl get endpoints <svc>` — if `<none>`, your selector is wrong or pods aren't `Ready`. `kubectl describe svc` shows the selector.
3. **Can the pod reach the port?** From a test pod: `nc -vz <svc> <port>`. If no, you're looking at NetworkPolicy or a CNI issue.

### "I can hit the Service but it goes to the wrong pod"

Usually:

- **Two Services with overlapping selectors.** A pod matches both, and you're seeing the wrong Service's pod.
- **Session affinity is wrong** for your protocol — set `sessionAffinity: ClientIP` if you need sticky.

### "External traffic isn't reaching the cluster"

For LoadBalancer Services: `kubectl get svc -o wide` — is the `EXTERNAL-IP` populated? If `<pending>`, your cloud LB controller hasn't provisioned it.

For Ingress: `kubectl describe ingress` — `Address` column should show the controller's LB. If empty, the controller isn't watching this Ingress's `IngressClass`.

### "NetworkPolicy isn't blocking anything"

- **Wrong CNI.** kindnet doesn't enforce — see notebook 09's table.
- **No policy selects the pods.** Pods without any matching policy fall back to allow-all.
- **The policy's `from` / `to` is too permissive.** Re-read the OR-list / AND-fields rule (notebook 09).

### "I get `connection refused` from outside"

Layered checks:

1. Is the node's IP reachable?
2. Is the NodePort open on the node?
3. Is the Service's selector matching pods? (`kubectl get ep`)
4. Are the pods listening on the right port? (`kubectl exec <pod> -- netstat -ln` or `ss -ln`)
5. Is the container's port the same as `containerPort` and `targetPort`?

### `kubectl run --rm tmp --image=nicolaka/netshoot -it -- bash`

The classic debug-pod-in-a-pod move. `netshoot` ships every networking tool you can think of: `dig`, `nslookup`, `tcpdump`, `mtr`, `curl`, `nc`, `iperf3`, `kubectl`, `ss`, `ip`. From inside it you can prod the cluster the way a misbehaving pod would.

In [ ]:
# kubectl explain — the offline API reference
!kubectl explain pod.spec.containers.resources --recursive 2>&1 | head -25
!echo '---'
# Generate YAML for a complex object you'd otherwise have to type
!kubectl create deployment demo --image=nginx:alpine --replicas=3 --dry-run=client -o yaml | head -30
!echo '---'
# auth can-i with impersonation — the fastest "did my RBAC work?" check
!kubectl auth can-i create deployments --as=system:serviceaccount:default:reader-sa
!kubectl auth can-i get pods --as=system:serviceaccount:default:reader-sa
!echo '---'
# kubectl top — needs metrics-server; on kind it usually isn't installed
!kubectl top nodes 2>&1 | head -5 || echo "(metrics-server not installed on this cluster)"
!echo '---'
# kubectl api-resources — every kind in this cluster, with its short name and group
!kubectl api-resources --namespaced=true -o wide 2>/dev/null | head -10

## CKA exam strategy

The CKA is 2 hours, 15 to 20 hands-on tasks, scored by automated test scripts. You're given access to a fleet of practice clusters in a browser terminal; you fix things.

### Time budget

- **Read the whole question** before typing. The "context switch" instruction at the top of each task (`kubectl config use-context <foo>`) is mandatory; missing it means working on the wrong cluster.
- **Skip-and-return on the hard ones.** Each question is worth a fixed number of points; spending 30 minutes on a 4-point task while leaving a 9-point task untouched is a losing trade. Mark "later," move on.
- **Save 10 minutes at the end** to re-check skipped items.

### Aliases you'll be glad you set

The exam allows you to set up your shell. The standard set:

```bash
alias k=kubectl
export do='--dry-run=client -o yaml'         # k run app --image=nginx $do
export now='--grace-period=0 --force'        # k delete pod foo $now
source <(kubectl completion bash)            # tab completion
complete -o default -F __start_kubectl k     # tab completion for the alias
```

### The single most important muscle memory

```bash
kubectl run x --image=nginx --dry-run=client -o yaml > x.yaml
vim x.yaml                                   # add what you need
kubectl apply -f x.yaml
```

Almost every "write a manifest" task is faster with this than with raw YAML. Same with `kubectl create deployment / service / configmap / secret / role / rolebinding / cronjob` plus `--dry-run=client -o yaml`.

### Imperative shortcuts you must know

```bash
kubectl run pod --image=nginx --port=80
kubectl create deployment d --image=nginx --replicas=3
kubectl expose deployment d --port=80 --target-port=8080
kubectl scale deployment d --replicas=5
kubectl set image deployment/d app=nginx:1.28
kubectl rollout undo deployment/d
kubectl create configmap c --from-literal=key=val
kubectl create secret generic s --from-literal=key=val
kubectl create role r --verb=get,list --resource=pods
kubectl create rolebinding rb --role=r --user=alice
kubectl taint nodes n key=val:NoSchedule
kubectl label node n disk=ssd
kubectl drain n --ignore-daemonsets
kubectl cordon / uncordon n
```

### Pre-built drill list

Things to be able to do in **under 5 minutes each** without referencing docs:

- Write a Deployment + Service from scratch.
- Add liveness / readiness probes to an existing Deployment.
- Create a ConfigMap from a literal and mount it as both env vars and a volume.
- Create a Role for "read pods in namespace X" and bind it to a ServiceAccount.
- Apply a NetworkPolicy that denies all ingress except from a labelled namespace.
- Take an etcd snapshot, then restore from it.
- Scale a Deployment, do a rolling update, undo it.
- Drain a node, cordon, evict, uncordon.
- Find why a pod is `Pending`, `CrashLoopBackOff`, or `ImagePullBackOff`.

### Reference allowed during the exam

You can browse `kubernetes.io/docs` and the GitHub repos for kubeadm, etcd, kube-scheduler, kube-controller-manager during the exam. **Don't rely on it for muscle memory** — searching docs eats time. Use it for syntax you genuinely don't remember.

### Common gotchas

- **Wrong namespace.** Half of failed answers are "you wrote it in `default` instead of `team-a`." Always check `-n` first.
- **Wrong context.** Each question's `use-context` is mandatory.
- **Selector / template mismatch.** A Deployment with mismatched selector and template labels won't deploy. Copy `app: foo` into both.
- **Forgetting `runAsNonRoot` defaults.** PSA tasks fail silently if the pod's image runs as root.
- **`kubectl delete pod` doesn't respect PDBs**; `kubectl drain` does. Use `drain` when the question asks about graceful disruption.

## You're at the end of the course

You've gone from "I have never run a pod" to a 10-chapter map of Kubernetes — from primitives to cluster operation. The exam tests a slice of this — but the slice is wide. Treat the practice clusters as the real exam: time-boxed, no second chances, all keyboard.

### What this notebook leaves behind

```bash
kubectl delete pod sa-demo hardened 2>/dev/null
kubectl delete serviceaccount demo-sa reader-sa
kubectl delete role pod-reader
kubectl delete rolebinding reader-sa-pod-reader reader-sa-view
```

### Beyond the CKA

If the CKA went well, the natural follow-ons:

- **CKAD** (Certified Kubernetes Application Developer) — workload-side, lighter cluster admin.
- **CKS** (Certified Kubernetes Security Specialist) — needs CKA first. Deep dive on the security context, PSA, audit logging, image policy, runtime security (Falco), and supply chain.
- **The Gateway API** — the future of L7 routing (notebook 09 peek).
- **Service meshes** — Istio, Linkerd, Cilium Service Mesh.
- **Multi-cluster** — Argo CD's app-of-apps, Cluster API, Karmada.

Everything you've learned about declarative state, controllers, label selectors, and the API server applies to all of them. Kubernetes is a *language* for distributed systems; new tools speak the same one.

Good luck with the exam.